# Running the dashboard locally

## Import dependencies

Refer to [README.md](README.md) to correctly create your Python virtual environment. 

After creating and activating the Python venv, running the next code cell should work without issue.

In [9]:
import os
import yaml
from pathlib import Path

from data_processing import load_existing_journals
from utils import ensure_dir

Most of the following cells will use the **Data Journals Dashboard** CLI helper `dj` to fetch and process data journals metadata and build the Hugo websites. 

In order to prevent overwriting any existing files, we will define a output directory in the `tests` folder of the repository's working dir:

In [2]:
PROJECT_ROOT = Path(os.getcwd()).parent
SCHEMA_PATH = PROJECT_ROOT / "metadata_schema" / "schema.yaml"
OUTPUT_DIR = PROJECT_ROOT / "tests" / "output"

ensure_dir(OUTPUT_DIR)  # Create OUTPUT_DIR if it does not exist

Test if the CLI helper is correctly installed in our environment.

In [3]:
print(SCHEMA_PATH)

/Users/thomas/Coding/projects/data-journals-dashboard/metadata_schema/schema.yaml


In [4]:
!dj 

Usage: dj [OPTIONS] COMMAND [ARGS]...

  Data Journal Dashboard CLI Helper.

Options:
  --help  Show this message and exit.

Commands:
  collect   Fetch or parse raw journal metadata from GitHub or a local...
  process   Process raw journal metadata by validating it against the...
  hugo      Generate Hugo static site content from processed journal data.
  export    Export data journal metadata to different file types.
  validate  Validate journals against metadata schema.


## Collecting journal metadata from GitHub

The dashboard’s primary data source are data journal metadata published under CC0 1.0 Universal on Zenodo and Github.

>Kindling, Maxi, and Dorothea Strecker. 2022. List of Data Journals (1.0). [https://doi.org/10.5281/zenodo.7082126](https://doi.org/10.5281/zenodo.7082126). 

To collect the `csv` data from GitHub we use the CLI helper's `collect` command and pass our `OUTPUT_DIR` path:

In [5]:
!dj collect --output_fpath $OUTPUT_DIR

Fetching data journal data from GitHub...
Saved raw CSV → /Users/thomas/Coding/projects/data-journals-dashboard/tests/output/data_journals.csv


We can inspect the downloaded `csv`:

In [6]:
!head -n 5 ../tests/output/data_journals.csv

"ISSN","journal_title","publisher","URL","data_journal_type"
"2574-5417","Big Earth Data","Taylor & Francis","https://www.tandfonline.com/journals/tbed20","mixed"
"1809-127X","Check List","Pensoft","https://checklist.pensoft.net/","pure"
"1698-0476","Arxius de Miscel·lània Zoològica","Museu de Ciències Naturals de Barcelona","https://museucienciesjournals.cat/en/amz","pure"
"1941-4927","NCHS Data Briefs","Centers for Disease Control and Preventions (CDC)","https://www.cdc.gov/nchs/products/databriefs.htm","pure"


## Metadata processing

We process the raw `csv` metadata from GitHub by adding additional metadata from the [Directory of Open Access Journal's public API](https://doaj.org/api/v4/docs) with the CLI helper's `process all` command:

In [7]:
!dj process all --help

Usage: dj process all [OPTIONS]

  Process all journals in one go.

Options:
  -i, --input_fpath PATH   Path to the journal metadata CSV file.  [default:
                           data/raw/data_journals.csv]
  -s, --schema_path PATH   Path to the journal metadata schema YAML file.
                           [default: metadata_schema/schema.yaml]
  -o, --output_fpath PATH  Path to save the processed journal collection YAML
                           file.  [default: data/data_journals.yaml]
  --help                   Show this message and exit.


In [8]:
!dj process all \
    --input_fpath ../tests/output/data_journals.csv \
    --output_fpath $OUTPUT_DIR \
    --schema_path $SCHEMA_PATH

Parsed 142 journals.
[1/142] Adding metadata from doaj.org to 2574-5417...
[2/142] Adding metadata from doaj.org to 1809-127X...
[3/142] Adding metadata from doaj.org to 1698-0476...
[4/142] Adding metadata from doaj.org to 1941-4927...
  No DOAJ entry found for 1941-4927.
[5/142] Adding metadata from doaj.org to 1758-0463...
[6/142] Adding metadata from doaj.org to 2709-4715...
[7/142] Adding metadata from doaj.org to 2572-4754...
[8/142] Adding metadata from doaj.org to 2732-5121...
[9/142] Adding metadata from doaj.org to 2363-4952...
[10/142] Adding metadata from doaj.org to 2059-481X...
[11/142] Adding metadata from doaj.org to 2296-7745...
[12/142] Adding metadata from doaj.org to 2054-345X...
[13/142] Adding metadata from doaj.org to 2032-6378...
  No DOAJ entry found for 2032-6378.
[14/142] Adding metadata from doaj.org to 1297-966X...
[15/142] Adding metadata from doaj.org to 0092-640X...
  No DOAJ entry found for 0092-640X.
[16/142] Adding metadata from doaj.org to 1314-2828.

The processed journal metadata is saved as a `.yaml` file that we can load and inspect:

In [ ]:
journal_collection = load_existing_journals(fpath=OUTPUT_DIR / "data_journals.yaml")

# Inspect the first journal of the collection with complete metadata
journal_collection[0]

{'id': 1,
 'is_active': True,
 'enrichment_source': 'doaj',
 'issn': '2574-5417',
 'journal_title': 'Big Earth Data',
 'publisher': 'Taylor & Francis',
 'url': 'https://www.tandfonline.com/journals/tbed20',
 'data_journal_type': 'mixed',
 'eissn': '2574-5417',
 'pissn': '2096-4471',
 'language': ['EN'],
 'oa_start': 2017,
 'boai': True,
 'publication_time_weeks': 10,
 'keywords': ['geography',
  'geology',
  'atmospheric science',
  'marine science',
  'earth system science',
  'big data studies'],
 'research_fields': ['Geography. Anthropology. Recreation', 'Geology'],
 'subject_codes': [{'code': 'G',
   'scheme': 'LCC',
   'term': 'Geography. Anthropology. Recreation'},
  {'code': 'QE1-996.5', 'scheme': 'LCC', 'term': 'Geology'}],
 'publisher_name': 'Taylor & Francis Group',
 'publisher_country': 'GB',
 'institution_name': 'The International Society for Digital Earth',
 'institution_country': 'CN',
 'review_process': ['Anonymous peer review'],
 'review_url': 'https://www.tandfonline.c

## Generate hugo

In [ ]:
!dj hugo generate 